# Discussion 9/03/2026  --  Solutions

In this discussion we practice getting a session on the the Shared Computing Cluster (SCC).

The general instructions for running on the SCC are available under General Resources on [Piazza](https://piazza.com/bu/fall2026/ds722/resources).

This notebook also serves as a brief introduction to Pytorch where you will build and train a neural network for classification.

## Goals for today

1. Get a working Jupyter session on the SCC.
2. Build and train a small neural network in [PyTorch](https://pytorch.org).
3. See where the three parts of DS722 -- **linear algebra**, **optimization**, and **probability/statistics** -- show up inside that network.
4. (Optional) Re-run the same code on a GPU.

## How this notebook works

You are not expected to know any PyTorch yet. Every task follows the same pattern:

> **Example** -- a short, complete, runnable cell doing the thing on a tiny made-up dataset. Run it and read it.
>
> **Your turn** -- the same thing for our real dataset, as a skeleton with hints. Fill in the blanks.

Each example contains everything you need so be sure to read each one before attempting the exercise that follows it.

You need the following Python modules:

- PyTorch
- Scikit-learn
- NumPy
- Matplotlib

These modules are already available in the SCC academic ML environment. See Part 0 for details on how to load the relevant Python environment.

---

# Part 0 -- Getting a session on the SCC

## One-time setup

You only have to do this **once**. If you have already done it, skip ahead.

You will need to setup how your SCC account uses the minicond package. First, go to [SCC OnDemand](https://scc-ondemand.bu.edu). At the top of the page, you will see a `Login Nodes` tab. Select `Login Nodes` and then select `scc1`. This will launch a linux terminal session on the login node `scc1`. Then:

1. Run the command `module load miniconda`
2. Run the command `setup_scc_condarc.sh`
3. Input the number corresponding to the relevant project (most likely `ds722`) and hit enter

## Starting a Jupyter notebook session

Next, we will launch a Jupyter notebook session on the SCC by following the below steps.

1. Upload this notebook to your directory `/projectnb/ds722/students/username`, replacing `username` with your BU username.

2. Log on to [SCC OnDemand](https://scc-ondemand.bu.edu) with your BU username and password.

3. From the **Interactive Apps** dropdown, select **Jupyter Notebook**.

4. Fill in the form:
    - Click **Select Module** and select **both**
        - `miniconda`
        - `academic-ml/spring-2026`
    - In the **pre-launch command** box, enter: `conda activate spring-2026-pyt`
    - You may select either the notebook or the lab interface.
    - **Working Directory**: the directory where this notebook lives, typically `/projectnb/ds722/students/username`
    - **Number of Hours**: do **NOT** ask for more than 12.
    - **Number of Cores**: select 1 (do **NOT** ask for more than 4).
    - **Number of GPUs**: select **0** for now. We come back to this in Part 5.
    - Click **Launch**.

5. Be patient while the session starts. When you are finished, **close the notebook and cancel the session** so the resources are freed for everyone else.

> Alternatively you may work using **VSCode Server** on the SCC. The setup is nearly identical; see the instructions on Piazza.

**Two things to remember all semester:**

- Only request a GPU if you are actually going to use one. GPUs are the most contended resource on the cluster.
- Always cancel your session when you are done. An idle session still holds everything you asked for.

## Check your session

Run the cell below. It should print a PyTorch version and tell you whether a GPU is visible.

If it fails with `ModuleNotFoundError`, your kernel is not the one you think it is: go back and check the modules and the pre-launch command you entered in the OnDemand form.

In [ ]:
import sys
import platform

import torch

print("Python executable :", sys.executable)
print("Python version    :", platform.python_version())
print("PyTorch version   :", torch.__version__)
print("Hostname          :", platform.node())
print("GPU available     :", torch.cuda.is_available())

**Discuss.** What hostname did you get? Is that a login node or a compute node, and how can you tell?

---

# Part 1 -- Why a neural network?

In the first lecture we used a neural network as the motivating example for the whole course. Recall the picture: an input layer, one or more hidden layers, and an output layer.

**Forward propagation** computes the activation of unit $i$ in layer $l$ as

$$
a_{i}^{(l)} = \sigma\left(\sum_{j} w_{ij}^{(l)} a_{j}^{(l-1)} + b_i^{(l)}\right),
$$

where $w_{ij}^{(l)}$ are the weights, $b_i^{(l)}$ the biases, and $\sigma$ a nonlinear activation function. Each unit forms a weighted sum of the previous layer's activations and passes it through $\sigma$.

That weighted sum is where the **linear algebra** comes in. In Lecture 2, we will see that the above summation formula represents a *matrix-vector product*, and that pushing a whole batch of samples through a layer at once turns out to be a *matrix-matrix product*. In this notebook, we use PyTorch to compute these products.

**Training** the network means computing the weights and biases that make the loss function $\mathcal{L}$ as small as possible. That is an **optimization** problem. The algorithm used in this notebook is gradient descent. It repeatedly adjusts each weight based on the direction that decreases the loss. We will consider other algorithms for optimization later in the semester.

The quantity that tells us which direction to move is the gradient. The gradient is computed by **back propagation**:

$$
\frac{\partial \mathcal{L}}{\partial w_{ij}^{(l)}} = \delta_i^{(l)} a_j^{(l-1)},
$$

where the error term $\delta_i^{(l)}$ is computed using the chain rule of calculus. PyTorch computes this for you when you call `loss.backward()`. We will review the chain rule of calculus and the mechancis of backward propagation later in this course.

Finally, our network outputs a number between 0 and 1 that we read as a **probability** that the tumor is benign. The loss function that we minimize is the **cross-entropy** loss

$$
\mathcal{L}(y, \hat{y}) = -\sum_{i} y_i \log(\hat{y}_i),
$$

where $y$ is the true label and $\hat{y}$ is the predicted label. Deciding what that probability means, and how much to trust it, is the **probability and statistics** part of the course.

This small example encapsulates all three parts of the course.

> **Notation.** The three formulas above were given in the first lecture. Everything else in this notebook is code.

---

# Part 2 -- The data

We use the Wisconsin breast cancer dataset, which ships with scikit-learn. The features are computed from a digitized image of a fine needle aspirate (FNA) of a breast mass and describe characteristics of the cell nuclei in the image. The task is to classify a tumor as **malignant (0)** or **benign (1)**.

This is **supervised learning** where we train a model with labelled data.

### Example -- loading the data

Run this cell. It imports everything we need for the whole notebook and loads the dataset.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the breast cancer dataset as a dataframe
dataset = load_breast_cancer(as_frame=True)

# X is a Pandas dataframe whose columns are the features
X = dataset.data
# y is a Pandas series with the class labels (0 = malignant, 1 = benign)
y = dataset.target

print("Number of samples :", X.shape[0])
print("Number of features:", X.shape[1])
print("Target names      :", dataset.target_names)

### Your turn -- explore the data

Before modelling anything, look at it. Answer the following, one per line of code.

**Hints.**

- The full description of the dataset, including what each feature means: `print(dataset.DESCR)`
- The list of feature names: `dataset.feature_names`
- The first five rows: `X.head()`
- Column names, data types, and how many non-null entries each column has: `X.info()`
- How many samples are in each class: `y.value_counts()`

**Discuss.** Is this dataset balanced? If you built a model that always predicted "benign" and never looked at the features, what accuracy would it get? What does that tell you about reporting accuracy on its own?

In [ ]:
# TODO: print the description and the feature names

In [ ]:
# TODO: look at the first few rows, the column info, and the class counts

### Example -- splitting and standardizing

We hold out part of the data to test on, and we rescale the features. Here is the whole procedure on a made-up array of 6 samples with 2 features, so you can see the shapes and the effect.

Note the two different scaler methods: `fit_transform` on the training data, `transform` on the test data.

In [ ]:
# A tiny made-up dataset: 6 samples, 2 features on very different scales
X_demo = np.array([[1.0, 500.0],
                   [2.0, 520.0],
                   [3.0, 480.0],
                   [4.0, 510.0],
                   [5.0, 495.0],
                   [6.0, 505.0]])
y_demo = np.array([0, 0, 0, 1, 1, 1])

# Split into 67% train / 33% test
X_demo_train, X_demo_test, y_demo_train, y_demo_test = train_test_split(
    X_demo, y_demo, test_size=0.33, random_state=10)

print("train shape:", X_demo_train.shape, " test shape:", X_demo_test.shape)
print("before scaling:\n", X_demo_train)

# Create the scaler, learn the mean and spread from the TRAINING data, and apply it
sc_demo = StandardScaler()
X_demo_train = sc_demo.fit_transform(X_demo_train)

# Apply the SAME scaler to the test data -- fit_transform would relearn it
X_demo_test = sc_demo.transform(X_demo_test)

print("after scaling:\n", X_demo_train)
print("column means after scaling:", X_demo_train.mean(axis=0))

Observe what the scaler did. Prior to scaling, the second feature was around 500 while the first was around 3. After scaling, both columns have mean 0 and comparable spread (unit variance).

This matters because of the weighted sum $\sum_j w_{ij}^{(l)} a_j^{(l-1)}$ from Part 1. A feature measured in the hundreds contributes hundreds of times more to that sum than a feature measured in ones. Standardizing puts the features on an equal footing before the network ever sees them.

### Your turn -- split and standardize the real data

**Hints.**

- `X` and `y` are Pandas objects; `train_test_split` wants NumPy arrays, so pass `X.to_numpy()` and `y.to_numpy()`.
- Use `test_size=0.20` for an 80/20 split and `random_state=10` so everyone in the room gets the same split.
- Name the outputs `X_train, X_test, y_train, y_test` -- the rest of the notebook assumes those names.
- Follow the example exactly for the scaler: `fit_transform` on train, `transform` on test.
- Check yourself: `X_train.shape` should be `(455, 30)` and `X_test.shape` should be `(114, 30)`.

**Discuss.** The scaler learns a mean and a spread from whatever data you fit it on. Why should it learn those from the training data only, and not from the test data as well?

In [ ]:
# TODO: split X and y into X_train, X_test, y_train, y_test

# TODO: create a StandardScaler, fit_transform the training features,
#       and transform the test features

---

# Part 3 -- The building blocks

Nearly every PyTorch program is made of the same five objects:

| Object | What it does |
| --- | --- |
| `Dataset` | fetches one sample at a time |
| `DataLoader` | groups samples into minibatches and shuffles them |
| `Module` | the model itself |
| loss function | measures how wrong a prediction is |
| optimizer | updates the weights to reduce the loss |

We discuss each component individually and then combine for our full neural network example. For each one, the example uses the same tiny toy dataset defined in the next cell. This allows you to see the whole API working end-to-end on something you can read at a glance.

### The toy dataset used by the examples

8 samples, 2 features, 2 classes. Nothing about it is interesting except that it is small.

In [ ]:
X_toy = np.array([[0.1, 0.2],
                  [0.2, 0.1],
                  [0.3, 0.2],
                  [0.2, 0.3],
                  [0.8, 0.9],
                  [0.9, 0.8],
                  [0.7, 0.9],
                  [0.9, 0.7]])
y_toy = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print(X_toy.shape, y_toy.shape)

## The `Dataset` class

A `Dataset` retrieves the features and labels one sample at a time. To make your own, your Dataset class inherits from `torch.utils.data.Dataset` and you override the following three methods:

- `__init__(self, ...)` runs once, when the object is created. This is where you convert your data to tensors.
- `__len__(self)` returns the number of samples.
- `__getitem__(self, index)` returns the sample at position `index`.

Once you have written those three, Python's `len(obj)` and `obj[i]` work on your object, and PyTorch knows how to iterate over it.

### Example -- a complete `Dataset`

In [ ]:
class ToyDataset(Dataset):
    def __init__(self, X, y):
        # NumPy defaults to float64 but PyTorch layers expect float32, so cast
        self.X = torch.from_numpy(X.astype(np.float32))
        # Cast the labels too, and add a trailing dimension so each label
        # has shape (1,) instead of () -- see the note below
        self.y = torch.from_numpy(y.astype(np.float32)).unsqueeze(1)
        self.len = self.X.shape[0]

    def __len__(self):
        return self.len

    def __getitem__(self, index):
        return self.X[index], self.y[index]


toydata = ToyDataset(X_toy, y_toy)

print("number of samples :", len(toydata))
print("sample 0          :", toydata[0])
print("features shape    :", toydata[0][0].shape)
print("label shape       :", toydata[0][1].shape)

**Why `.unsqueeze(1)`?** Our network produces one number per sample, so a batch of 4 predictions has shape `(4, 1)`. The loss function compares predictions with labels elementwise, so the labels need that same shape. Without `unsqueeze(1)` they would have shape `(4,)` and the comparison would not line up.

### Your turn -- create `WisconsinDataset`

Write the same class for our data.

**Hints.**

- Copy the example and rename it. The bodies of all three methods are identical.
- `__init__` takes the already-split, already-scaled arrays, e.g. `WisconsinDataset(X_train, y_train)`.
- Then instantiate `traindata = WisconsinDataset(X_train, y_train)` and check it with the cell below.

**Discuss.** `__getitem__` returns one sample at a time. Why is that a sensible design when the dataset is 500 rows, and why is it *essential* when the dataset is 50 million images?

In [ ]:
class WisconsinDataset(Dataset):
    def __init__(self, X_train, y_train):
        # TODO
        pass

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass

In [ ]:
# Check your work -- uncomment once WisconsinDataset is written.
# Expected: 455 samples, features of shape torch.Size([30]), label of shape torch.Size([1])

# traindata = WisconsinDataset(X_train, y_train)
# print("number of samples :", len(traindata))
# print("features shape    :", traindata[0][0].shape)
# print("label shape       :", traindata[0][1].shape)

## The `DataLoader` class

A `DataLoader` wraps a `Dataset` and hands the model **minibatches** rather than single samples. It can also reshuffle the data every epoch, which reduces overfitting, and fetch batches in parallel.

### Example -- iterating over minibatches

In [ ]:
toyloader = DataLoader(toydata, batch_size=2)

for batch_number, (inputs, targets) in enumerate(toyloader):
    print("batch", batch_number, "| inputs", inputs.shape, "| targets", targets.shape)

Eight samples in batches of two gives four batches, each of shape `(2, 2)`: two samples, two features.

> *Preview of Lecture 2.* Stacking several samples into one array like this is what makes a **matrix-matrix product** the natural operation for a layer. It is why batching is fast, and it is one of the first genuinely practical payoffs of linear algebra you will see.

### Your turn -- create the training dataloader

**Hints.**

- Wrap `traindata`, use `batch_size = 4`, and call the result `trainloader`.
- Print the shapes of the first batch the same way the example does. You can grab a single batch with `next(iter(trainloader))`.

**Discuss.** With 455 training samples and a batch size of 4, how many batches are in one epoch? What are the dimensions of the array of inputs in a single batch?

In [ ]:
batch_size = 4

# TODO: create trainloader and print the shapes of one batch

## The `Module` class

`torch.nn.Module` is how you define a model. You inherit from this class it and override two methods:

- `__init__(self, ...)` creates the layers,
- `forward(self, x)` implements forward progagation (how data moves through the model).

An `nn.Linear(in_features, out_features)` layer computes exactly the weighted-sum-plus-bias from Part 1 -- it holds the $w_{ij}$ and the $b_i$ and applies them. The activation function $\sigma$ is applied separately, by calling something like `torch.relu` (rectified linear unit) or `torch.sigmoid` (sigmoid function that maps a real number to the range $(0, 1)$) on the result.

Layer types are documented at [https://pytorch.org/docs/stable/nn.html](https://pytorch.org/docs/stable/nn.html) and activations at [https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity](https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity).

### Example -- a complete one-layer model

This model takes 2 features straight to 1 output. No hidden layer.

In [ ]:
class ToyNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ToyNet, self).__init__()
        self.linear1 = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        x = torch.sigmoid(self.linear1(x))
        return x


torch.manual_seed(42)
toynet = ToyNet(2, 1)
print(toynet)

# Push one batch through the model
inputs, targets = next(iter(toyloader))
print("input :", inputs.shape)
print("output:", toynet(inputs).shape)
print(toynet(inputs))

Two samples went in, two numbers between 0 and 1 came out. Since the model is untrained, those numbers are meaningless but we have a functioning implementation.

### Your turn -- write `NeuralNetwork`

Build a network with **one hidden layer**: 30 features in, `hidden_layer_dim` units in the middle, 1 output.

**Hints.**

- You need two layers, not one. In `__init__`, create `self.linear1 = nn.Linear(input_dim, hidden_layer_dim)` and a second layer `self.linear2` going from `hidden_layer_dim` to `output_dim`.
- In `forward`, apply `self.linear1`, then `torch.relu`, then `self.linear2`, then `torch.sigmoid`, returning the result. Each line looks like `x = torch.relu(self.linear1(x))`.
- Keep `super(NeuralNetwork, self).__init__()` as the first line of `__init__` or nothing will work.
- Then instantiate `clf` and print it, as in the example.

**Discuss.** Why a sigmoid on the *output* layer specifically? Look back at the cross-entropy formula in Part 1 and think about what $\log(\hat{y})$ does if $\hat{y}$ is allowed to be 0 or negative.

In [ ]:
# Number of features (columns of X_train)
input_dim = X_train.shape[1]

# Width of the hidden layer
hidden_layer_dim = 4

# Number of outputs
output_dim = 1

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_layer_dim, output_dim):
        super(NeuralNetwork, self).__init__()
        # TODO

    def forward(self, x):
        # TODO
        pass

In [ ]:
# Set a random seed so everyone starts from the same weights
torch.manual_seed(42)

# TODO: create clf = NeuralNetwork(...), print it, and push one batch through it

### Example -- looking inside the model

The weights and biases are just tensors, and you can access them.

In [ ]:
[w, b] = toynet.linear1.parameters()

print("weight shape:", w.shape)
print("bias shape  :", b.shape)
print("weights     :", w.data)
print("total number of parameters:", sum(p.numel() for p in toynet.parameters()))

### Your turn -- count the parameters of your model

**Hints.**

- Do the same for `clf.linear1.parameters()` and `clf.linear2.parameters()`.
- Work out the total by hand from the shapes first: each `nn.Linear(a, b)` holds a weight of shape `(b, a)` and a bias of shape `(b,)`.
- Then check with `sum(p.numel() for p in clf.parameters())`.

**Discuss.** Where do almost all of the parameters live -- the first layer or the second? What happens to that count if you change `hidden_layer_dim` from 4 to 64?

In [ ]:
# TODO

## Loss function and optimizer

The **loss function** measures how far a prediction is from the truth. Which one you use depends on what the model predicts:

- Regression: mean squared error, `nn.MSELoss`
- Binary classification: binary cross-entropy, `nn.BCELoss`

`nn.BCELoss` is the two-class case of the cross-entropy from Part 1. With only two classes, the sum $-\sum_i y_i \log(\hat{y}_i)$ has two terms, and since the two probabilities must add to 1, it becomes

$$
\mathcal{L}(y, \hat{y}) = -\left[\, y \log \hat{y} + (1-y)\log(1-\hat{y}) \,\right]
$$

for a single sample with true label $y$ equal to 0 or 1. Read it in two halves: if the true label is 1, only the first term survives and the loss is $-\log \hat y$, which is 0 when the model says 1 and grows without bound as the model approaches 0. If the true label is 0, the second term does the same job in reverse.

The **optimizer** updates the weights to reduce the loss. We use stochastic gradient descent (`torch.optim.SGD`), which repeatedly subtracts a small multiple of the gradient from each weight. That multiple is the **learning rate**: too small and training crawls, too large and it overshoots. The other available optimizers are listed at [https://pytorch.org/docs/stable/optim.html](https://pytorch.org/docs/stable/optim.html).

### Example -- creating both

In [ ]:
loss_function_toy = nn.BCELoss()
optimizer_toy = torch.optim.SGD(toynet.parameters(), lr=0.1)

# What does the loss look like on one batch, before any training?
inputs, targets = next(iter(toyloader))
outputs = toynet(inputs)
print("predictions:", outputs.detach().numpy().ravel())
print("targets    :", targets.numpy().ravel())
print("loss       :", loss_function_toy(outputs, targets).item())

### Your turn -- create the loss and optimizer for your model

**Hints.**

- `loss_function = nn.BCELoss()`
- `optimizer = torch.optim.SGD(clf.parameters(), lr=0.01)`
- The optimizer needs `clf.parameters()` so it knows which tensors it is allowed to change.

In [ ]:
# TODO

---

# Part 4 -- Training

Training alternates **forward propagation** (data passes through the model, producing predictions) with **backward propagation** (gradients are computed and the weights are updated). One full pass (forward + backward propagation) over the training data is an **epoch**.

### Example -- a complete training loop

Every PyTorch training loop you will ever write is built from these steps in this order. Read them carefully; you are about to write them again.

Note the structure: the counters are reset at the top of each epoch, so the numbers we print describe *that* epoch. Inside, we go batch by batch.

In [ ]:
torch.manual_seed(42)
toynet = ToyNet(2, 1)
loss_function_toy = nn.BCELoss()
optimizer_toy = torch.optim.SGD(toynet.parameters(), lr=0.1)

# Per-epoch history, for plotting later
toy_losses, toy_accuracies = [], []

for epoch in range(5):

    # Reset the counters at the start of every epoch
    correct, total = 0, 0
    running_loss = 0.0

    for data in toyloader:
        # 1. unpack the minibatch
        inputs, targets = data

        # 2. forward propagation
        outputs = toynet(inputs)

        # 3. compute the loss for this batch
        loss = loss_function_toy(outputs, targets)

        # 4. clear the gradients left over from the previous step
        optimizer_toy.zero_grad()

        # 5. backward propagation: compute the gradients
        loss.backward()

        # 6. update the weights
        optimizer_toy.step()

        # 7. bookkeeping: anything above 0.5 counts as a prediction of class 1
        predicted = torch.round(outputs.data)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()
        running_loss += loss.item()

    # Average the loss over the batches, and compute the accuracy for this epoch
    epoch_loss = running_loss / len(toyloader)
    epoch_acc = correct / total

    toy_losses.append(epoch_loss)
    toy_accuracies.append(epoch_acc)
    print("epoch {}  loss : {:.5f}  accuracy : {:.5f}".format(epoch, epoch_loss, epoch_acc))

Step 4 is very important but is often overlooked. Be aware that PyTorch *accumulates* gradients by default. As a result, without `zero_grad()` the gradients from earlier batches are still stored, and each update would use this stale information to compute an incorrect gradient.

### Your turn -- train your network

**Hints.**

- The structure is exactly the example's. Change `toynet` to `clf`, `toyloader` to `trainloader`, `loss_function_toy` to `loss_function`, `optimizer_toy` to `optimizer`.
- Keep the resets at the top of the epoch loop and the appends at the bottom, outside the batch loop.
- 10 epochs at `lr=0.01`. It takes a few seconds.

**Discuss.** Watch the loss and accuracy as they print. Are they both moving in the direction you would hope? Does the improvement per epoch get bigger or smaller as training goes on?

In [ ]:
epochs = 10

# Per-epoch history
losses = []
accuracies = []

for epoch in range(epochs):

    # TODO: reset correct, total and running_loss for this epoch

    for data in trainloader:
        # TODO: steps 1-7 from the example
        pass

    # TODO: compute epoch_loss and epoch_acc, append them to the lists,
    #       and print the epoch summary

### Example -- plotting the training history

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)
fig.set_size_inches(10, 4)

ax1.plot(toy_losses)
ax1.set_title('Loss vs Epochs')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')

ax2.plot(toy_accuracies)
ax2.set_title('Accuracy vs Epochs')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')

plt.tight_layout()
plt.show()

### Your turn -- plot your training history

**Hints.**

- Same two-panel figure, using `losses` and `accuracies` instead of the toy lists.
- Make it readable: title each panel and label both axes.

**Discuss.** Was 10 epochs enough? Reading the two curves, what would you change first -- more epochs, a different learning rate -- and what in the plot makes you say so?

In [ ]:
# TODO

### Example -- evaluating on held-out data

To measure a model honestly you run it on data it never trained on, and you turn off gradient tracking while you do it. There is no need to compute gradients when you are making predictions (often called inference).

In [ ]:
toy_testdata = ToyDataset(X_toy, y_toy)   # standing in for a real test set
toy_testloader = DataLoader(toy_testdata, batch_size=2)

correct, total = 0, 0

# No gradients needed when we are only making predictions
with torch.no_grad():
    for data in toy_testloader:
        inputs, labels = data
        outputs = toynet(inputs)
        predicted = torch.round(outputs.data)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Toy accuracy: {:.1f} %".format(100 * correct / total))

`torch.no_grad()` tells PyTorch to stop recording the information it would need to compute gradients. With no calls to `loss.backward()` here, recording it wastes memory and time.

### Your turn -- evaluate on the test set

**Hints.**

- Build `testdata = WisconsinDataset(X_test, y_test)` and wrap it in a `DataLoader` called `testloader` with the same `batch_size`.
- Copy the evaluation loop, swapping in `clf` and `testloader`.
- Report the accuracy as a percentage.

**Discuss.** How does the test accuracy compare with the training accuracy from your last epoch? Which of the two is the honest description of how good this model is, and why? (Look back at your answer to the class-balance question in Part 2 before you decide whether to be impressed.)

In [ ]:
# TODO

---

# Part 5 -- Optional: run it on a GPU

Everything so far ran on a **CPU**, the general-purpose processor that executes the instructions of whatever software is running on the machine. As models and datasets grow, a CPU takes longer and longer to get through the arithmetic inside forward and backward propagation, and training times grow with it.

A **GPU** (graphics processing unit) is hardware built to do very many simple arithmetic operations at the same time. That happens to be exactly the shape of the computation in a neural network layer, where thousands of independent weighted sums are evaluated at once. For this reason, the SCC’s 400-plus GPUs are its most sought-after resource.

Adapting the code is mostly a matter of sending the model and each minibatch to the GPU.

## Getting a GPU session

Feel free to test this section out at a later time, not everyone in the discussion will be able to get a GPU at the same time. You can also work on this section in pairs.

Note that on the SCC if your GPU is idle for more than 2 hours, the job is killed by the GPU-reaper. The SCC does not allow idle GPUs since it is a waste of resources.

Cancel your current session, then start a new Jupyter session exactly as in Part 0, with one change:

- Set **Number of GPUs** to **1**. Do **not** ask for more than 1.

Be prepared to wait -- GPU nodes are busy, and your session may sit in the queue. Only ask for one if you are actually going to run this section, and cancel the session as soon as you are finished.

Then re-run the notebook from the top before continuing here.

### Example -- finding and using a device

A `device` is just a label saying where a tensor or model lives. The code snippet below is the standard one to set the device to either a CPU or GPU. CUDA stands for Compute Unified Device Architecture and is a parallel computing platform and application programming interface (API) created by NVIDIA for general purpose programming on GPUs (not just graphics).

In [ ]:
# torch.cuda.is_available() is True when a GPU is visible to PyTorch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Code is running on:", "GPU" if device.type == "cuda" else "CPU")

# Moving a tensor: .to() RETURNS a new tensor, it does not modify in place
t = torch.ones(3)
print("before:", t.device)
t = t.to(device)
print("after :", t.device)

# Moving a model: modules ARE modified in place, no reassignment needed
toynet.to(device)
print("model :", next(toynet.parameters()).device)

**One detail worth pausing on.** Tensors and modules behave differently when you move them:

```python
inputs = inputs.to(device)   # tensors: .to() returns a new tensor, so reassign
toynet.to(device)            # modules: moved in place, no reassignment needed
```

Always reassign when moving a tensor. If you ever want to confirm where something is, print `inputs.device`.

### Your turn -- train on the GPU

**Hints.**

- Set up the device with the `if` block from the example, and print it. If it says CPU, your session did not get a GPU -- check the OnDemand form.
- Create a fresh model, `clf2 = NeuralNetwork(input_dim, hidden_layer_dim, output_dim)`, plus its own `loss_function2` and `optimizer2`. Seed with `torch.manual_seed(42)` first so it starts from the same weights as `clf`.
- Send the model to the device: `clf2.to(device)`.
- Copy your training loop from Part 4 and add two lines right after unpacking the batch, moving `inputs` and `targets` to the device. Remember to reassign: `inputs = inputs.to(device)`.

**Discuss.** Compare the printed losses with your CPU run. Since both models started from the same seed, do the numbers agree?

In [ ]:
# TODO: set up the device and print which one you got

In [ ]:
torch.manual_seed(42)

# TODO: create clf2, loss_function2, optimizer2, and send the model to the device

In [ ]:
# TODO: the training loop, with each minibatch moved to the device

### Was it actually faster?

Almost certainly not -- and the reason is worth more than the speedup would have been.

A GPU is a **throughput** device: it has thousands of cores and wins by keeping them all busy at once. Our model has 129 parameters and our batches hold 4 samples, so the largest operation in the entire training loop multiplies a $4 \times 30$ array by a $30 \times 4$ one. That is a few thousand multiply-adds. This work finishes on a single CPU core before the GPU has finished being *told* to start.

Meanwhile every batch pays overhead the CPU version never pays: copying `inputs` and `targets` across to the device, and a separate kernel launch for each operation in the forward and backward pass. Though it may not seem like a lot, a launch costs microseconds, which is orders of magnitude larger than a few thousand arithmetic operations.

So "is the GPU faster?" has no fixed answer. It depends on how much arithmetic each step contains, and we can go and find the crossover.

**Why we need different data for this.** The lever is **arithmetic per step**, which is set by the batch size and the width of the layers -- *not* by how many rows the dataset has. More rows alone just means more iterations, not bigger ones, so the overhead scales up right alongside the work. But the two are linked: you cannot use a batch of 1024 when you only have 455 training samples. So below we generate a larger dataset with scikit-learn's `make_classification`, purely so that large batches become possible.

### Example -- timing a training run properly

In [ ]:
import time

from sklearn.datasets import make_classification
from torch.utils.data import TensorDataset


def make_loader(n_samples, n_features, batch, seed=0):
    Xb, yb = make_classification(n_samples=n_samples, n_features=n_features,
                                 n_informative=10, random_state=seed)
    Xb = torch.from_numpy(Xb.astype(np.float32))
    yb = torch.from_numpy(yb.astype(np.float32)).unsqueeze(1)
    return DataLoader(TensorDataset(Xb, yb), batch_size=batch)


def time_one_epoch(device, loader, n_features, hidden):
    torch.manual_seed(42)
    model = NeuralNetwork(n_features, hidden, 1).to(device)
    lf = nn.BCELoss()
    opt = torch.optim.SGD(model.parameters(), lr=0.01)

    if device.type == "cuda":
        torch.cuda.synchronize()     # don't start the clock mid-queue
    start = time.perf_counter()

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        loss = lf(model(inputs), targets)
        opt.zero_grad()
        loss.backward()
        opt.step()

    if device.type == "cuda":
        torch.cuda.synchronize()     # GPU work is queued; wait for it before stopping
    return time.perf_counter() - start


# 200,000 samples with 64 features, in batches of 1024
big_loader = make_loader(n_samples=200_000, n_features=64, batch=1024)
print("batches per epoch:", len(big_loader))

cpu = torch.device("cpu")
print("hidden=4  on CPU: {:.2f} s".format(time_one_epoch(cpu, big_loader, 64, hidden=4)))

The two `torch.cuda.synchronize()` calls are the point of that helper. GPU work is queued asynchronously -- Python races ahead while the card is still busy -- so timing without them measures how fast you *submitted* the work, not how long it took. You get a suspiciously excellent number.

### Your turn -- find where the GPU takes over

Keep the dataset and the batch size fixed, and widen the model. For each hidden width in `[4, 64, 512, 2048]`, time one epoch on the CPU and one on the GPU, and print the speedup.

**Hints.**

- Reuse `big_loader` and `time_one_epoch` from the example. Nothing new needs writing.
- `gpu = torch.device("cuda")`, and guard the whole thing with `if torch.cuda.is_available():` so the cell still runs if you did not get a GPU.
- Speedup is `cpu_time / gpu_time`. Above 1 means the GPU won.
- The whole sweep takes well under a minute. The `hidden=2048` case is the slow one on CPU.

**Discuss.**

1. At which hidden width does the GPU overtake the CPU, and what happens to the speedup as the model gets wider?
2. The number of *rows* never changed in this experiment, and neither did the number of batches. Why was widening the model enough to flip the result?
3. Try `hidden=2048` again with `batch=32` instead of 1024 (build a second loader). The model is just as wide, so why does the GPU do so much worse?

In [ ]:
# TODO: sweep the hidden width, timing CPU vs GPU for each

In [ ]:
# TODO: question 3 -- the same wide model with a much smaller batch

---

# Wrapping up

Before you leave:

- **Cancel your SCC session**, especially if you requested a GPU.
- Save your notebook in `/projectnb/ds722/students/username`, and download a copy if you want one locally.

## What to take away

You trained a neural network, and in doing so you touched all three parts of this course:

- The forward pass is a chain of weighted sums and activation functions. Those weighted sums are matrix operations. **Linear algebra**, starting in Lecture 2.
- Training is the minimization of a loss function by gradient descent. **Optimization**, in the middle third of the course.
- The output is a probability and the loss is a cross-entropy. **Probability and statistics**, in the final third.

We also observed how the compute hardware can have an effect on the speed of your computation. However, just naively moving your computation to potentially faster hardware does not automatically result in faster computing times.